### Research Question 2

**Which pathologies does the model treat similarly?**

In [4]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.neighbors import NearestNeighbors

In [2]:
# Import data from the base notebook
import import_ipynb
from IPython.utils.capture import capture_output

In [3]:
# Use "capture" do suppress plotting images
with capture_output() as captured:
    %run xraymodel.ipynb

### Compute correlations of model's predictions and show in heatmap

In [ ]:
pathology_names = XRAY_DATASET.pathologies
predictions_df = pd.DataFrame(
    MODEL_SCORE_PREDICTIONS_CONVERTED,
    columns=pathology_names
)

correlation_matrix = predictions_df.corr(method="pearson")

In [19]:
def plot_similarity_as_heatmap(correlation_matrix):
    fig = px.imshow(
        correlation_matrix,
        x=pathology_names,
        y=pathology_names,
        text_auto=".2f",
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        aspect="auto",
        title="Correlation Between Pathology Predictions"
    )

    fig.update_layout(
        xaxis_title="Pathology",
        yaxis_title="Pathology",
        # Make the heatmap less weird shaped!
        width=1100,
        height=900,
        xaxis_tickangle=-45
    )


    fig.show()

In [20]:
plot_similarity_as_heatmap(correlation_matrix)

### Observation

The heatmap shows  that some pathologies are treated similarly. For example, Atelictrasis and Lung Opacity seem often predicted together. Its seems that there is a small group that is often predicted together, containing Lung Opacity, Effusion, Atelectrasis, Consolidation and Edema.

### Compute correlations of ground-truth and show in heatmap

In [12]:
predictions_ground_truth_df = pd.DataFrame(
    XRAY_LABELS_LIST,
    columns=pathology_names
)

correlation_matrix_ground_truth = predictions_ground_truth_df.corr(method="pearson")

In [13]:
plot_similarity_as_heatmap(correlation_matrix_ground_truth)

### Observation

The correlation between the ground-truth pathology labels is weaker than for the model's prediction score, but it also shows that some pathologies actually occur more often together which is also correctly identified by the model. For exmaple, the group with Lung Opacity, Effusion, Atelectrasis, Consolidation and Edema also forms a cluster in the ground-truth heatmap, altought there correlation is much weaker. 

One interpretation: The model correctly classifies a group of pathologies that occur often together, but fails to distinguish them.

### Compute difference correlations and show in heatmap

In [15]:
correlation_difference = (
    correlation_matrix - correlation_matrix_ground_truth
)

fig = px.imshow(
    correlation_difference,
    x=pathology_names,
    y=pathology_names,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="Difference: Model Correlation vs. Ground Truth"
)

fig.show()

### Observation

The difference shows that the model's predictions are more correlated than the ground-truth, since all values ( except diagonal) are positive.

A strong correlation that only occurs in the model's prediction is Pneumonia, which has several cells with high positive values.

### Investigate high correlations

In [36]:
def show_correlation(path1, path2, threshold=0.7):


    scatter_df = pd.DataFrame({
        f"{path1} Score": predictions_df[path1],
        f"{path2} Score": predictions_df[path2],
        path1: predictions_ground_truth_df[path1].astype(int),
        path2: predictions_ground_truth_df[path2].astype(int),
    })

    scatter_df["Image Index"] = scatter_df.index

    conditions = [
        (scatter_df[path1] == 0) & (scatter_df[path2] == 0),
        (scatter_df[path1] == 1) & (scatter_df[path2] == 0),
        (scatter_df[path1] == 0) & (scatter_df[path2] == 1),
        (scatter_df[path1] == 1) & (scatter_df[path2] == 1)
    ]

    categories = [
        "Neither pathology",
        f"{path1} only",
        f"{path2} only",
        "Both pathologies"
    ]

    scatter_df["Ground Truth Combination"] = pd.Series(
        pd.Categorical(
            pd.Series(
                np.select(conditions, categories, default="Unknown"),
                index=scatter_df.index
            ),
            categories=categories
        )
    )

    symbol_map = {
        "Neither pathology": "circle",
        f"{path1} only": "square",
        f"{path2} only": "triangle-up",
        "Both pathologies": "diamond"
    }

    color_map = {
        "Neither pathology": "#636EFA",
        f"{path1} only": "#EF553B",
        f"{path2} only": "#00CC96",
        "Both pathologies": "#AB63FA"
    }

    fig = px.scatter(
        scatter_df,
        x=f"{path1} Score",
        y=f"{path2} Score",
        color="Ground Truth Combination",
        symbol="Ground Truth Combination",
        symbol_map=symbol_map,
        color_discrete_map=color_map,
        hover_data=["Image Index"],
        category_orders={"Ground Truth Combination": categories},
        title=f"Model Predictions: {path1} vs. {path2}",
        labels={
            f"{path1} Score": f"{path1} Prediction Score",
            f"{path2} Score": f"{path2} Prediction Score",
            "Ground Truth Combination": "Actual Labels"
        }
    )

    fig.update_traces(
        marker=dict(size=7, opacity=0.7)
    )

    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_color="black",
        annotation_text=f"{path2} threshold"
    )

    fig.add_vline(
        x=threshold,
        line_dash="dash",
        line_color="black",
        annotation_text=f"{path1} threshold"
    )

    fig.update_layout(
        width=1100,
        height=800,
        legend_title="Ground Truth",
        template="plotly_white"
    )

    fig.show()

In [37]:
show_correlation("Lung Lesion", "Pneumonia")

In [38]:
threshold = 0.7

# Binary predictions
pred_ll = (predictions_df["Lung Lesion"] >= threshold).astype(int)
pred_pneu = (predictions_df["Pneumonia"] >= threshold).astype(int)

# Ground truth
true_ll = predictions_ground_truth_df["Lung Lesion"].astype(int)
true_pneu = predictions_ground_truth_df["Pneumonia"].astype(int)

# Error analysis
results = pd.DataFrame({
    "Lung Lesion FP": (pred_ll == 1) & (true_ll == 0),
    "Lung Lesion FN": (pred_ll == 0) & (true_ll == 1),
    "Pneumonia FP": (pred_pneu == 1) & (true_pneu == 0),
    "Pneumonia FN": (pred_pneu == 0) & (true_pneu == 1)
})

print(results.sum())

both_false_positive = (
    (pred_ll == 1) &
    (pred_pneu == 1) &
    (true_ll == 0) &
    (true_pneu == 0)
)

print("Both predicted positive, neither actually present:",
      both_false_positive.sum())

Lung Lesion FP    1142
Lung Lesion FN     525
Pneumonia FP      1175
Pneumonia FN       388
dtype: int64
Both predicted positive, neither actually present: 808


### Observation

Lung Lesion and Pneumonia increase indeed together, altough the relationship is not perfectly linear. There are a lot of false positives in the right top area. 

The model may respond to image patterns that influence both pathology scores.